In [1]:
import pandas as pd
import numpy as np
import os

DATA_DIR = r"C:\Users\ASUS\Desktop\1º ano m\IACH\projeto\eicu-collaborative-research-database-demo-2.0.1\data"

patient = pd.read_csv(
    os.path.join(DATA_DIR, "patient.csv.gz"),
    low_memory=False
)

vitalPeriodic = pd.read_csv(
    os.path.join(DATA_DIR, "vitalPeriodic.csv.gz"),
    low_memory=False
)

vitalAperiodic = pd.read_csv(
    os.path.join(DATA_DIR, "vitalAperiodic.csv.gz"),
    low_memory=False
)

print("Files loaded successfully.")

Files loaded successfully.


In [5]:
print("Shape:", vitalPeriodic.shape)
print("Columns:")
print(vitalPeriodic.columns.tolist())

vitalPeriodic.info()
display(vitalPeriodic.head())

# First 24h
vitalPeriodic_24h = vitalPeriodic[
    (vitalPeriodic["observationoffset"] >= 0) &
    (vitalPeriodic["observationoffset"] <= 1440)
].copy()

print(
    "Vital observations first 24h:",
    len(vitalPeriodic_24h)
)

print(
    "ICU stays with vitals first 24h:",
    vitalPeriodic_24h["patientunitstayid"].nunique()
)

vital_cols = [
    "temperature",
    "sao2",
    "heartrate",
    "respiration",
    "cvp",
    "etco2",
    "systemicsystolic",
    "systemicdiastolic",
    "systemicmean",
    "pasystolic",
    "padiastolic",
    "pamean",
    "icp"
]

# Coverage
total_stays = vitalPeriodic_24h["patientunitstayid"].nunique()

print("\nCoverage by vital:")
for col in vital_cols:

    n_stays = vitalPeriodic_24h.loc[
        vitalPeriodic_24h[col].notna(),
        "patientunitstayid"
    ].nunique()

    print(
        f"{col}: {n_stays}/{total_stays} "
        f"({n_stays/total_stays*100:.1f}%)"
    )

# Missing values
missing_vitals = pd.DataFrame({
    "missing": vitalPeriodic_24h[vital_cols].isna().sum(),
    "missing_pct": (
        vitalPeriodic_24h[vital_cols].isna().mean() * 100
    ).round(2)
})

display(
    missing_vitals.sort_values(
        "missing_pct"
    )
)

# Duplicates
print(
    "Duplicated rows:",
    vitalPeriodic_24h.duplicated().sum()
)

# Basic statistics
display(
    vitalPeriodic_24h[vital_cols].describe().T
)

# Measurements per ICU stay
measurements_per_stay = (
    vitalPeriodic_24h
    .groupby("patientunitstayid")
    .size()
)

print(
    "\nMeasurements per ICU stay:"
)

print(
    measurements_per_stay.describe()
)

Shape: (1634960, 19)
Columns:
['vitalperiodicid', 'patientunitstayid', 'observationoffset', 'temperature', 'sao2', 'heartrate', 'respiration', 'cvp', 'etco2', 'systemicsystolic', 'systemicdiastolic', 'systemicmean', 'pasystolic', 'padiastolic', 'pamean', 'st1', 'st2', 'st3', 'icp']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1634960 entries, 0 to 1634959
Data columns (total 19 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   vitalperiodicid    1634960 non-null  int64  
 1   patientunitstayid  1634960 non-null  int64  
 2   observationoffset  1634960 non-null  int64  
 3   temperature        112820 non-null   float64
 4   sao2               1441700 non-null  float64
 5   heartrate          1627894 non-null  float64
 6   respiration        1381351 non-null  float64
 7   cvp                202284 non-null   float64
 8   etco2              72780 non-null    float64
 9   systemicsystolic   226475 non-null   float64
 10 

,vitalperiodicid,patientunitstayid,observationoffset,temperature,sao2,heartrate,respiration,cvp,etco2,systemicsystolic,systemicdiastolic,systemicmean,pasystolic,padiastolic,pamean,st1,st2,st3,icp
0,29524122,141765,1179,NaN,NaN,82.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,29557845,141765,189,NaN,97.0,76.0,30.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,29524442,141765,1169,NaN,NaN,84.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,29513052,141765,1534,NaN,NaN,92.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,29524600,141765,1164,NaN,NaN,86.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Vital observations first 24h: 553846
ICU stays with vitals first 24h: 2370

Coverage by vital:
temperature: 152/2370 (6.4%)
sao2: 2320/2370 (97.9%)
heartrate: 2359/2370 (99.5%)
respiration: 2151/2370 (90.8%)
cvp: 251/2370 (10.6%)
etco2: 101/2370 (4.3%)
systemicsystolic: 399/2370 (16.8%)
systemicdiastolic: 398/2370 (16.8%)
systemicmean: 401/2370 (16.9%)
pasystolic: 81/2370 (3.4%)
padiastolic: 81/2370 (3.4%)
pamean: 81/2370 (3.4%)
icp: 7/2370 (0.3%)


,missing,missing_pct
heartrate,2790,0.50
sao2,65790,11.88
respiration,67275,12.15
systemicmean,477446,86.21
systemicsystolic,478090,86.32
systemicdiastolic,478092,86.32
cvp,505045,91.19
temperature,523972,94.61
pamean,536219,96.82
pasystolic,536433,96.86


Duplicated rows: 0


,count,mean,std,min,25%,50%,75%,max
temperature,29874.0,39.058584,10.872285,0.5,36.7,37.2,37.8,102.4
sao2,488056.0,96.495037,5.001862,0.0,95.0,97.0,99.0,100.0
heartrate,551056.0,84.349554,18.996591,0.0,70.0,83.0,96.0,202.0
respiration,486571.0,19.504786,6.530600,0.0,16.0,19.0,23.0,163.0
cvp,48801.0,32.504621,75.104926,-50.0,7.0,11.0,16.0,400.0
etco2,15303.0,26.237797,13.392947,0.0,21.0,28.0,36.0,80.0
systemicsystolic,75756.0,120.399071,24.262010,-44.0,105.0,119.0,135.0,349.0
systemicdiastolic,75754.0,59.171186,16.320534,-76.0,50.0,58.0,66.0,344.0
systemicmean,76400.0,79.764699,20.195172,-45.0,68.0,77.0,88.0,346.0
pasystolic,17413.0,34.427612,11.733408,3.0,27.0,33.0,40.0,291.0



Measurements per ICU stay:
count    2370.000000
mean      233.690295
std        78.614261
min         1.000000
25%       209.250000
50%       275.000000
75%       287.000000
max       289.000000
dtype: float64


In [6]:

print("Shape:", vitalAperiodic.shape)
print("\nColumns:")
print(vitalAperiodic.columns.tolist())

vitalAperiodic.info()

display(vitalAperiodic.head())

#primeiras 24 horas


vitalAperiodic_24h = vitalAperiodic[
    (vitalAperiodic["observationoffset"] >= 0) &
    (vitalAperiodic["observationoffset"] <= 1440)
].copy()

print("\nAperiodic observations first 24h:", len(vitalAperiodic_24h))

print(
    "ICU stays with aperiodic vitals first 24h:",
    vitalAperiodic_24h["patientunitstayid"].nunique()
)

#coverage

aperiodic_cols = [
    col for col in vitalAperiodic_24h.columns
    if col not in [
        "vitalaperiodicid",
        "patientunitstayid",
        "observationoffset"
    ]
]

total_stays_ap = vitalAperiodic_24h["patientunitstayid"].nunique()

print("\nCoverage by aperiodic vital:")

for col in aperiodic_cols:

    n_stays = vitalAperiodic_24h.loc[
        vitalAperiodic_24h[col].notna(),
        "patientunitstayid"
    ].nunique()

    print(
        f"{col}: {n_stays}/{total_stays_ap} "
        f"({n_stays/total_stays_ap*100:.1f}%)"
    )


#missing values
missing_ap = pd.DataFrame({
    "missing": vitalAperiodic_24h[aperiodic_cols].isna().sum(),
    "missing_pct": (
        vitalAperiodic_24h[aperiodic_cols].isna().mean() * 100
    ).round(2)
})

display(
    missing_ap.sort_values("missing_pct")
)

#duplicates

print(
    "Duplicated rows:",
    vitalAperiodic_24h.duplicated().sum()
)

display(
    vitalAperiodic_24h[aperiodic_cols].describe().T
)


measurements_per_stay_ap = (
    vitalAperiodic_24h
    .groupby("patientunitstayid")
    .size()
)

print("\nAperiodic measurements per ICU stay:")
print(measurements_per_stay_ap.describe())

Shape: (274088, 13)

Columns:
['vitalaperiodicid', 'patientunitstayid', 'observationoffset', 'noninvasivesystolic', 'noninvasivediastolic', 'noninvasivemean', 'paop', 'cardiacoutput', 'cardiacinput', 'svr', 'svri', 'pvr', 'pvri']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 274088 entries, 0 to 274087
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   vitalaperiodicid      274088 non-null  int64  
 1   patientunitstayid     274088 non-null  int64  
 2   observationoffset     274088 non-null  int64  
 3   noninvasivesystolic   241808 non-null  float64
 4   noninvasivediastolic  241823 non-null  float64
 5   noninvasivemean       243178 non-null  float64
 6   paop                  169 non-null     float64
 7   cardiacoutput         14235 non-null   float64
 8   cardiacinput          11865 non-null   float64
 9   svr                   17789 non-null   float64
 10  svri                  3440 non

,vitalaperiodicid,patientunitstayid,observationoffset,noninvasivesystolic,noninvasivediastolic,noninvasivemean,paop,cardiacoutput,cardiacinput,svr,svri,pvr,pvri
0,3661418,141764,81,171.0,90.0,116.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3661424,141764,334,153.0,78.0,103.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3661417,141764,77,176.0,87.0,107.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3661419,141764,165,173.0,106.0,128.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3661421,141764,255,182.0,103.0,133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Aperiodic observations first 24h: 109601
ICU stays with aperiodic vitals first 24h: 2315

Coverage by aperiodic vital:
noninvasivesystolic: 2310/2315 (99.8%)
noninvasivediastolic: 2310/2315 (99.8%)
noninvasivemean: 2311/2315 (99.8%)
paop: 15/2315 (0.6%)
cardiacoutput: 47/2315 (2.0%)
cardiacinput: 34/2315 (1.5%)
svr: 33/2315 (1.4%)
svri: 28/2315 (1.2%)
pvr: 15/2315 (0.6%)
pvri: 14/2315 (0.6%)


,missing,missing_pct
noninvasivemean,12721,11.61
noninvasivediastolic,13267,12.10
noninvasivesystolic,13274,12.11
cardiacoutput,101049,92.20
cardiacinput,103403,94.34
svr,104281,95.15
svri,106365,97.05
paop,109493,99.90
pvr,109493,99.90
pvri,109494,99.90


Duplicated rows: 0


,count,mean,std,min,25%,50%,75%,max
noninvasivesystolic,96327.0,117.521048,25.208053,0.0,100.00,115.0,132.0,269.00
noninvasivediastolic,96334.0,64.797548,16.693836,0.0,54.00,63.0,74.0,244.00
noninvasivemean,96880.0,80.065751,18.440773,0.0,68.00,78.0,90.0,248.00
paop,108.0,15.351852,5.117890,5.0,11.75,15.0,19.0,36.00
cardiacoutput,8552.0,1.774464,2.759212,0.0,0.00,0.0,4.8,13.40
cardiacinput,6198.0,0.106255,0.513017,0.0,0.00,0.0,0.0,4.23
svr,5320.0,972.763910,436.454352,-3478.0,722.75,947.5,1160.0,6200.00
svri,3236.0,1713.123300,796.629995,-8313.0,1317.75,1594.0,2141.0,11408.00
pvr,108.0,110.333333,50.976263,25.0,75.00,100.0,133.0,302.00
pvri,107.0,198.775701,81.353214,0.0,150.00,188.0,240.0,432.00



Aperiodic measurements per ICU stay:
count    2315.000000
mean       47.343844
std        82.124814
min         1.000000
25%        21.000000
50%        34.000000
75%        58.000000
max      1500.000000
dtype: float64


## variáveis escolhidas:
- Heart rate -> vitalPeriodic

- SpO₂ -> vitalPeriodic

- Respiratory rate -> vitalPeriodic

- Systolic BP -> vitalAperiodic

- Diastolic BP -> vitalAperiodic

- Mean BP -> vitalAperiodic

- Temperature	-> ainda por decidir/investigar

- **excluimos:** paop, cardiacoutput, cardiacinput, svr, svri, pvr, pvri pois têm uma cobertura demasiado baixa cerca de 0.6% e 2%.

# ============================================================
# QUALITY CHECK - VARIABLES SELECTED FOR THE VITAL AGENT
# ============================================================

In [7]:
periodic_vars = [
    "heartrate",
    "sao2",
    "respiration",
    "temperature"
]

aperiodic_vars = [
    "noninvasivesystolic",
    "noninvasivediastolic",
    "noninvasivemean"
]

## 1. Cobertura e missing values

In [8]:
def quality_summary(df, variables, name):

    total_stays = df["patientunitstayid"].nunique()

    rows = []

    for col in variables:

        stays_with_data = df.loc[
            df[col].notna(),
            "patientunitstayid"
        ].nunique()

        rows.append({
            "variable": col,
            "ICU_stays_with_data": stays_with_data,
            "coverage_pct": f"{stays_with_data / total_stays * 100:.2f}%",
            "missing_values": df[col].isna().sum(),
            "missing_pct": f"{df[col].isna().mean() * 100:.2f}%"
        })

    result = pd.DataFrame(rows)

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    display(result)


quality_summary(
    vitalPeriodic_24h,
    periodic_vars,
    "VITAL PERIODIC"
)

quality_summary(
    vitalAperiodic_24h,
    aperiodic_vars,
    "VITAL APERIODIC"
)


VITAL PERIODIC


,variable,ICU_stays_with_data,coverage_pct,missing_values,missing_pct
0,heartrate,2359,99.54%,2790,0.50%
1,sao2,2320,97.89%,65790,11.88%
2,respiration,2151,90.76%,67275,12.15%
3,temperature,152,6.41%,523972,94.61%



VITAL APERIODIC


,variable,ICU_stays_with_data,coverage_pct,missing_values,missing_pct
0,noninvasivesystolic,2310,99.78%,13274,12.11%
1,noninvasivediastolic,2310,99.78%,13267,12.10%
2,noninvasivemean,2311,99.83%,12721,11.61%


## 2. Estatísticas e percentis

In [9]:
print("PERIODIC VARIABLES")

display(
    vitalPeriodic_24h[periodic_vars]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .T
)


print("APERIODIC VARIABLES")

display(
    vitalAperiodic_24h[aperiodic_vars]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .T
)

PERIODIC VARIABLES


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
heartrate,551056.0,84.349554,18.996591,0.0,47.000,57.0,70.0,83.0,96.0,118.0,133.0,202.0
sao2,488056.0,96.495037,5.001862,0.0,85.000,91.0,95.0,97.0,99.0,100.0,100.0,100.0
respiration,486571.0,19.504786,6.530600,0.0,5.000,11.0,16.0,19.0,23.0,31.0,39.0,163.0
temperature,29874.0,39.058584,10.872285,0.5,32.889,35.5,36.7,37.2,37.8,39.1,98.6,102.4


APERIODIC VARIABLES


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
noninvasivesystolic,96327.0,117.521048,25.208053,0.0,68.0,82.0,100.0,115.0,132.0,163.0,188.0,269.0
noninvasivediastolic,96334.0,64.797548,16.693836,0.0,32.0,42.0,54.0,63.0,74.0,94.0,113.0,244.0
noninvasivemean,96880.0,80.065751,18.440773,0.0,44.0,55.0,68.0,78.0,90.0,113.0,133.0,248.0


## 3. Quantas medições existem por paciente?

In [10]:
def measurements_per_patient(df, variables):

    results = []

    for col in variables:

        counts = (
            df
            .dropna(subset=[col])
            .groupby("patientunitstayid")[col]
            .count()
        )

        results.append({
            "variable": col,
            "patients": len(counts),
            "mean_measurements": round(counts.mean(), 1),
            "median_measurements": round(counts.median(), 1),
            "min_measurements": counts.min(),
            "max_measurements": counts.max()
        })

    return pd.DataFrame(results)


display(
    measurements_per_patient(
        vitalPeriodic_24h,
        periodic_vars
    )
)

display(
    measurements_per_patient(
        vitalAperiodic_24h,
        aperiodic_vars
    )
)

,variable,patients,mean_measurements,median_measurements,min_measurements,max_measurements
0,heartrate,2359,233.6,275.0,1,289
1,sao2,2320,210.4,250.0,1,289
2,respiration,2151,226.2,267.0,1,289
3,temperature,152,196.5,224.5,1,289


,variable,patients,mean_measurements,median_measurements,min_measurements,max_measurements
0,noninvasivesystolic,2310,41.7,33.0,1,287
1,noninvasivediastolic,2310,41.7,33.0,1,287
2,noninvasivemean,2311,41.9,33.0,1,287


## 4. Procurar valores problemáticos

In [11]:
quality_flags = {
    "heartrate": (
        (vitalPeriodic_24h["heartrate"] <= 0)
    ),

    "sao2": (
        (vitalPeriodic_24h["sao2"] < 0) |
        (vitalPeriodic_24h["sao2"] > 100)
    ),

    "respiration": (
        (vitalPeriodic_24h["respiration"] <= 0)
    ),

    "noninvasivesystolic": (
        vitalAperiodic_24h["noninvasivesystolic"] <= 0
    ),

    "noninvasivediastolic": (
        vitalAperiodic_24h["noninvasivediastolic"] <= 0
    ),

    "noninvasivemean": (
        vitalAperiodic_24h["noninvasivemean"] <= 0
    )
}


print("Potentially invalid values:\n")

for variable, condition in quality_flags.items():

    print(
        variable,
        "→",
        condition.sum()
    )

Potentially invalid values:

heartrate → 147
sao2 → 0
respiration → 3754
noninvasivesystolic → 27
noninvasivediastolic → 37
noninvasivemean → 30


## 5. Temperature — analisar separadamente

In [12]:
temp = vitalPeriodic_24h["temperature"].dropna()

print("Temperature observations:", len(temp))

print("\nBasic statistics:")
print(temp.describe())

print("\nValues <= 50:")
print((temp <= 50).sum())

print("\nValues > 50:")
print((temp > 50).sum())

Temperature observations: 29874

Basic statistics:
count    29874.000000
mean        39.058584
std         10.872285
min          0.500000
25%         36.700000
50%         37.200000
75%         37.800000
max        102.400000
Name: temperature, dtype: float64

Values <= 50:
28868

Values > 50:
1006


In [13]:
print("\nTemperature <= 50:")
print(
    temp[temp <= 50]
    .describe(
        percentiles=[0.01, 0.05, 0.95, 0.99]
    )
)

print("\nTemperature > 50:")
print(
    temp[temp > 50]
    .describe(
        percentiles=[0.01, 0.05, 0.95, 0.99]
    )
)


Temperature <= 50:
count    28868.000000
mean        37.047219
std          1.342568
min          0.500000
1%          32.889000
5%          35.400000
50%         37.200000
95%         38.400000
99%         39.400000
max         42.900000
Name: temperature, dtype: float64

Temperature > 50:
count    1006.000000
mean       96.776342
std         3.313984
min        90.500000
1%         90.700000
5%         91.700000
50%        97.800000
95%       101.400000
99%       102.200000
max       102.400000
Name: temperature, dtype: float64


In [14]:
print(
    vitalPeriodic_24h[
        ["patientunitstayid",
         "observationoffset",
         "temperature"]
    ]
    .dropna()
    .sort_values("temperature")
    .head(20)
)

print(
    vitalPeriodic_24h[
        ["patientunitstayid",
         "observationoffset",
         "temperature"]
    ]
    .dropna()
    .sort_values(
        "temperature",
        ascending=False
    )
    .head(20)
)

         patientunitstayid  observationoffset  temperature
2644                151179                474         0.50
2861                151179               1264         8.70
2888                151179                489         8.75
2656                151179                464        10.40
2753                151179               1269        13.50
1477891            3116577               1000        13.90
34784               197619                684        14.30
1477599            3116577               1005        14.40
1477883            3116577               1020        15.10
1477251            3116577               1010        15.30
1477287            3116577               1015        16.00
34724               197619                734        16.00
1204355            2727377                137        16.75
2914                151179                289        17.00
2938                151179                294        17.60
3196                151179                314        17.

### temperatura standardized and cleaned, em celsius

In [15]:
vitalPeriodic_24h["temperature_celsius"] = vitalPeriodic_24h["temperature"].copy()

# Convert likely Fahrenheit values to Celsius
mask_f = vitalPeriodic_24h["temperature_celsius"] > 50

vitalPeriodic_24h.loc[mask_f, "temperature_celsius"] = (
    vitalPeriodic_24h.loc[mask_f, "temperature_celsius"] - 32
) * 5 / 9

print(vitalPeriodic_24h["temperature_celsius"].describe())

count    29874.000000
mean        37.011512
std          1.375650
min          0.500000
25%         36.600000
50%         37.200000
75%         37.700000
max         42.900000
Name: temperature_celsius, dtype: float64


In [16]:
vitalPeriodic_24h.loc[
    (vitalPeriodic_24h["temperature_celsius"] < 25) |
    (vitalPeriodic_24h["temperature_celsius"] > 45),
    "temperature_celsius"
] = pd.NA

In [17]:
print(vitalPeriodic_24h["temperature_celsius"].describe())

print(
    "Missing after cleaning:",
    vitalPeriodic_24h["temperature_celsius"].isna().sum()
)

count    29825.000000
mean        37.040354
std          1.159825
min         25.400000
25%         36.600000
50%         37.200000
75%         37.700000
max         42.900000
Name: temperature_celsius, dtype: float64
Missing after cleaning: 524021


In [18]:
temp_stays = vitalPeriodic_24h.loc[
    vitalPeriodic_24h["temperature_celsius"].notna(),
    "patientunitstayid"
].nunique()

total_stays = vitalPeriodic_24h["patientunitstayid"].nunique()

print("ICU stays with valid temperature:", temp_stays)

print(
    "Coverage:",
    f"{temp_stays / total_stays * 100:.2f}%"
)

ICU stays with valid temperature: 152
Coverage: 6.41%


ficamos com 2 colunas:

temperature
→ original

temperature_celsius
→ standardized and cleaned


temos apenas 152 pacientes com temperaturas validas, vamos colocar esta variavel como opcional

# ============================================================
# FINAL DATASET - VITAL AGENT
# First 24h: mean, min, max, last
# ============================================================

In [19]:
# Sort by time so that "last" really corresponds
# to the last available measurement
vitalPeriodic_24h = vitalPeriodic_24h.sort_values(
    ["patientunitstayid", "observationoffset"]
)

vitalAperiodic_24h = vitalAperiodic_24h.sort_values(
    ["patientunitstayid", "observationoffset"]
)


# ------------------------------------------------------------
# 1. PERIODIC VITALS
# ------------------------------------------------------------

periodic_summary = (
    vitalPeriodic_24h
    .groupby("patientunitstayid")
    .agg(
        hr_mean=("heartrate", "mean"),
        hr_min=("heartrate", "min"),
        hr_max=("heartrate", "max"),
        hr_last=("heartrate", "last"),

        sao2_mean=("sao2", "mean"),
        sao2_min=("sao2", "min"),
        sao2_max=("sao2", "max"),
        sao2_last=("sao2", "last"),

        rr_mean=("respiration", "mean"),
        rr_min=("respiration", "min"),
        rr_max=("respiration", "max"),
        rr_last=("respiration", "last"),

        temp_mean=("temperature_celsius", "mean"),
        temp_min=("temperature_celsius", "min"),
        temp_max=("temperature_celsius", "max"),
        temp_last=("temperature_celsius", "last")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 2. APERIODIC VITALS - BLOOD PRESSURE
# ------------------------------------------------------------

aperiodic_summary = (
    vitalAperiodic_24h
    .groupby("patientunitstayid")
    .agg(
        sbp_mean=("noninvasivesystolic", "mean"),
        sbp_min=("noninvasivesystolic", "min"),
        sbp_max=("noninvasivesystolic", "max"),
        sbp_last=("noninvasivesystolic", "last"),

        dbp_mean=("noninvasivediastolic", "mean"),
        dbp_min=("noninvasivediastolic", "min"),
        dbp_max=("noninvasivediastolic", "max"),
        dbp_last=("noninvasivediastolic", "last"),

        map_mean=("noninvasivemean", "mean"),
        map_min=("noninvasivemean", "min"),
        map_max=("noninvasivemean", "max"),
        map_last=("noninvasivemean", "last")
    )
    .reset_index()
)

In [20]:
# ------------------------------------------------------------
# 3. MERGE
# ------------------------------------------------------------

vital_agent_data = (
    patient[["patientunitstayid"]]
    .drop_duplicates()
    .merge(
        periodic_summary,
        on="patientunitstayid",
        how="left"
    )
    .merge(
        aperiodic_summary,
        on="patientunitstayid",
        how="left"
    )
)

print("Shape:", vital_agent_data.shape)

display(vital_agent_data.head())

Shape: (2520, 29)


,patientunitstayid,hr_mean,hr_min,hr_max,hr_last,sao2_mean,sao2_min,sao2_max,sao2_last,rr_mean,...,sbp_max,sbp_last,dbp_mean,dbp_min,dbp_max,dbp_last,map_mean,map_min,map_max,map_last
0,141764,106.652174,90.0,138.0,104.0,NaN,NaN,NaN,NaN,NaN,...,182.0,153.0,93.375000,78.0,106.0,78.0,125.625000,103.0,143.0,103.0
1,141765,83.190972,72.0,116.0,88.0,96.608696,95.0,98.0,97.0,24.782609,...,165.0,156.0,70.217391,59.0,87.0,85.0,95.173913,75.0,127.0,127.0
2,143870,45.449367,40.0,55.0,49.0,96.284810,80.0,100.0,95.0,67.031646,...,117.0,107.0,53.166667,50.0,59.0,54.0,74.500000,68.0,80.0,78.0
3,144815,83.800000,68.0,118.0,104.0,98.630682,96.0,100.0,99.0,16.927273,...,136.0,124.0,72.777778,46.0,85.0,70.0,88.777778,61.0,103.0,93.0
4,145427,65.457983,49.0,85.0,75.0,95.761905,89.0,100.0,98.0,16.349206,...,147.0,132.0,62.833333,55.0,79.0,71.0,83.208333,72.0,105.0,95.0


### verificação final

In [21]:
print("Number of ICU stays:", len(vital_agent_data))

print(
    "Unique ICU stays:",
    vital_agent_data["patientunitstayid"].nunique()
)

print(
    "Duplicated patientunitstayid:",
    vital_agent_data["patientunitstayid"].duplicated().sum()
)

display(
    vital_agent_data.isna()
    .mean()
    .mul(100)
    .round(2)
    .to_frame("missing_pct")
)

Number of ICU stays: 2520
Unique ICU stays: 2520
Duplicated patientunitstayid: 0


,missing_pct
patientunitstayid,0.00
hr_mean,6.39
hr_min,6.39
hr_max,6.39
hr_last,6.39
sao2_mean,7.94
sao2_min,7.94
sao2_max,7.94
sao2_last,7.94
rr_mean,14.64


In [22]:
display(vital_agent_data.head())

print("\nColumns:")
print(vital_agent_data.columns.tolist())

print("\nBasic statistics:")
display(vital_agent_data.describe().T)

,patientunitstayid,hr_mean,hr_min,hr_max,hr_last,sao2_mean,sao2_min,sao2_max,sao2_last,rr_mean,...,sbp_max,sbp_last,dbp_mean,dbp_min,dbp_max,dbp_last,map_mean,map_min,map_max,map_last
0,141764,106.652174,90.0,138.0,104.0,NaN,NaN,NaN,NaN,NaN,...,182.0,153.0,93.375000,78.0,106.0,78.0,125.625000,103.0,143.0,103.0
1,141765,83.190972,72.0,116.0,88.0,96.608696,95.0,98.0,97.0,24.782609,...,165.0,156.0,70.217391,59.0,87.0,85.0,95.173913,75.0,127.0,127.0
2,143870,45.449367,40.0,55.0,49.0,96.284810,80.0,100.0,95.0,67.031646,...,117.0,107.0,53.166667,50.0,59.0,54.0,74.500000,68.0,80.0,78.0
3,144815,83.800000,68.0,118.0,104.0,98.630682,96.0,100.0,99.0,16.927273,...,136.0,124.0,72.777778,46.0,85.0,70.0,88.777778,61.0,103.0,93.0
4,145427,65.457983,49.0,85.0,75.0,95.761905,89.0,100.0,98.0,16.349206,...,147.0,132.0,62.833333,55.0,79.0,71.0,83.208333,72.0,105.0,95.0



Columns:
['patientunitstayid', 'hr_mean', 'hr_min', 'hr_max', 'hr_last', 'sao2_mean', 'sao2_min', 'sao2_max', 'sao2_last', 'rr_mean', 'rr_min', 'rr_max', 'rr_last', 'temp_mean', 'temp_min', 'temp_max', 'temp_last', 'sbp_mean', 'sbp_min', 'sbp_max', 'sbp_last', 'dbp_mean', 'dbp_min', 'dbp_max', 'dbp_last', 'map_mean', 'map_min', 'map_max', 'map_last']

Basic statistics:


,count,mean,std,min,25%,50%,75%,max
patientunitstayid,2520.0,1.666226e+06,1.022224e+06,141764.000000,772262.750000,1.590372e+06,2.697422e+06,3.353113e+06
hr_mean,2359.0,8.427343e+01,1.702708e+01,28.272727,71.955474,8.280800e+01,9.584805e+01,1.531200e+02
hr_min,2359.0,6.816151e+01,1.906704e+01,0.000000,58.000000,6.800000e+01,8.000000e+01,1.420000e+02
hr_max,2359.0,1.073260e+02,2.256799e+01,47.000000,92.000000,1.050000e+02,1.220000e+02,2.020000e+02
hr_last,2359.0,8.331539e+01,2.016076e+01,0.000000,71.000000,8.200000e+01,9.500000e+01,1.890000e+02
sao2_mean,2320.0,9.630220e+01,3.484019e+00,41.015748,95.161333,9.674013e+01,9.817259e+01,1.000000e+02
sao2_min,2320.0,8.571121e+01,1.398528e+01,0.000000,83.000000,9.000000e+01,9.300000e+01,1.000000e+02
sao2_max,2320.0,9.936595e+01,1.418219e+00,71.000000,99.000000,1.000000e+02,1.000000e+02,1.000000e+02
sao2_last,2320.0,9.536293e+01,7.742971e+00,0.000000,95.000000,9.700000e+01,9.900000e+01,1.000000e+02
rr_mean,2151.0,1.951638e+01,5.119714e+00,0.000000,16.303603,1.877912e+01,2.200720e+01,6.703165e+01


# guardar datasets finais:

In [23]:
# Raw temporal data needed by the Vital Agent

vitalPeriodic_agent_raw = vitalPeriodic_24h[
    [
        "patientunitstayid",
        "observationoffset",
        "heartrate",
        "sao2",
        "respiration",
        "temperature_celsius"
    ]
].copy()

vitalAperiodic_agent_raw = vitalAperiodic_24h[
    [
        "patientunitstayid",
        "observationoffset",
        "noninvasivesystolic",
        "noninvasivediastolic",
        "noninvasivemean"
    ]
].copy()

In [24]:
OUTPUT_DIR = os.path.join(
    os.getcwd(),
    "processed_data"
)

os.makedirs(OUTPUT_DIR,exist_ok=True)

vital_agent_data.to_csv(
    os.path.join(OUTPUT_DIR, "vital_agent_data.csv"),
    index=False
)

vitalPeriodic_agent_raw.to_csv(
    os.path.join(OUTPUT_DIR, "vitalPeriodic_agent_raw.csv"),
    index=False
)

vitalAperiodic_agent_raw.to_csv(
    os.path.join(OUTPUT_DIR, "vitalAperiodic_agent_raw.csv"),
    index=False
)